In [5]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from google.colab import files

# 1. 데이터 파일 자동 확인 및 업로드 처리
DATA_PATH = 'knhanes_like_week4.csv'

if not os.path.exists(DATA_PATH):
    print("데이터 파일이 없습니다. 'knhanes_like_week4.csv' 파일을 업로드해주세요.")
    uploaded = files.upload()

# 데이터 로드
df = pd.read_csv(DATA_PATH)

# 2. 변수 전처리 (읍면 거주: urban == 2 -> 1, 그외 -> 0)
df['urban_rural'] = (df['urban'] == 2).astype(float)
df['obes'] = (df['bmi'] >= 25).astype(float)

# B-3 시연을 위한 충돌부 'prog' 생성 (재현 가능하도록 seed 고정)
_rng = np.random.default_rng(20260922)
_p = 1 / (1 + np.exp(-(-1.6 + 1.5 * df['obes'] + 1.5 * df['htn'])))
df['prog'] = (_rng.random(len(df)) < _p).astype(float)

# 분석에 필요한 변수와 설계 변수 선택 후 결측치 제거
cols_team = ['htn', 'urban_rural', 'age', 'edu', 'income', 'sbp', 'prog', 'wt', 'kstrata', 'psu']
df_team = df[cols_team].dropna()

w_t  = df_team['wt'].to_numpy(float)
ST_t = df_team['kstrata'].to_numpy()
PS_t = df_team['psu'].to_numpy()

print(f"정상 분석 대상 데이터 수: {len(df_team):,}명")

# 3. 복합표본 설계를 반영한 svyglm 함수 정의 (샌드위치 표준오차)
def _vcl(U, strata, psu):
    t = pd.DataFrame(U)
    t['st'] = strata
    t['cl'] = psu
    uc = t.groupby(['st', 'cl']).sum(numeric_only=True)
    p = U.shape[1]
    B = np.zeros((p, p))
    for s, g in uc.groupby(level=0):
        m = len(g)
        if m > 1:
            D = g.values - g.values.mean(axis=0)
            B += (m / (m - 1)) * (D.T @ D)
    return B

def svyglm(y, X, w, strata, psu, family='binomial'):
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    w = np.asarray(w, float)
    if family == 'binomial':
        m = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = mu * (1 - mu)
    else:
        m = sm.GLM(y, X, family=sm.families.Gaussian(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = np.ones_like(mu)
    b = m.params
    A = X.T @ np.diag(w * v) @ X
    U = (w * (y - mu))[:, None] * X
    B = _vcl(U, strata, psu)
    Ai = np.linalg.inv(A)
    V  = Ai @ B @ Ai
    return b, np.sqrt(np.diag(V)), m

def or_ci(b, se, i=1):
    return np.exp(b[i]), np.exp(b[i] - 1.96 * se[i]), np.exp(b[i] + 1.96 * se[i])


# --- [B-2] 무보정 모형 및 보정 모형(연령·교육·소득) ---
X_un = sm.add_constant(df_team[['urban_rural']].astype(float))
b_un, se_un, _ = svyglm(df_team['htn'], X_un, w_t, ST_t, PS_t)
or_u, l_u, h_u = or_ci(b_un, se_un, i=1)

cols_adj = ['urban_rural', 'age', 'edu', 'income']
X_ad = sm.add_constant(df_team[cols_adj].astype(float))
b_ad, se_ad, _ = svyglm(df_team['htn'], X_ad, w_t, ST_t, PS_t)
or_a, l_a, h_a = or_ci(b_ad, se_ad, i=1)


# --- [B-3] 과보정 모형 (+ 수축기혈압 sbp 추가) ---
cols_over = ['urban_rural', 'age', 'edu', 'income', 'sbp']
X_ov = sm.add_constant(df_team[cols_over].astype(float))
b_ov, se_ov, _ = svyglm(df_team['htn'], X_ov, w_t, ST_t, PS_t)
or_o, l_o, h_o = or_ci(b_ov, se_ov, i=1)


# 결과 테이블 정리 및 출력
result_table = pd.DataFrame({
    'Model': ['무보정 모형 (Unadjusted)', '보정 모형 (Adjusted: 연령·교육·소득)', '과보정 모형 (Overadjusted: + 수축기혈압)'],
    'Exposure': ['urban_rural', 'urban_rural', 'urban_rural'],
    'OR (95% CI)': [
        f"{or_u:.3f} ({l_u:.3f} - {h_u:.3f})",
        f"{or_a:.3f} ({l_a:.3f} - {h_a:.3f})",
        f"{or_o:.3f} ({l_o:.3f} - {h_o:.3f})"
    ]
})

print("\n[팀 과제 B-2 및 B-3 회귀분석 결과 비교]")
print(result_table.to_string(index=False))

정상 분석 대상 데이터 수: 5,367명

[팀 과제 B-2 및 B-3 회귀분석 결과 비교]
                         Model    Exposure           OR (95% CI)
           무보정 모형 (Unadjusted) urban_rural 1.692 (1.439 - 1.989)
    보정 모형 (Adjusted: 연령·교육·소득) urban_rural 1.018 (0.833 - 1.245)
과보정 모형 (Overadjusted: + 수축기혈압) urban_rural 0.941 (0.786 - 1.126)
